In [ ]:
import sys
sys.path.append("../..")

import pandas as pd
import numpy as np
import json
from python.functions.bridge import parse_quarter, build_bridge_inputs, run_bridge

In [ ]:
target = 1_500_000

B = build_bridge_inputs()

rdl  = run_bridge("../../data/outputs/forecasts/obr_scenario_forecasts.csv", "ensemble_log",
                   B["seed"], B["p"], B["seasonal"], B["fy_map"], B["net_add"], B["actual_back"])
vecm = run_bridge("../../data/outputs/forecasts/vecm_unconditional_forecast.csv", "vecm_log",
                   B["seed"], B["p"], B["seasonal"], B["fy_map"], B["net_add"], B["actual_back"], strip_space=True)

for name, d in [("ARDL/NARDL ensemble", rdl), ("VECM (unconditional)", vecm)]:
    print(name)
    for fy, v in d.items():
        print(f"{fy}: {v:,.0f}")
    cumulative = sum(d.values())
    print(f"Cumulative: {cumulative:,.0f} ({100*cumulative/target:.1f}% of {target:,}, "
          f"shortfall {target-cumulative:,.0f})\n")


non-private new build is held flat at its recent average (it won't respond to the policy scenarios), and conversions/change-of-use/demolitions are likewise held at 2021-24 averages. 

In [ ]:
import matplotlib.pyplot as plt

fy_start = lambda s: int(str(s)[:4])  # "2022-23" -> 2022

# Actual history from LT120, plus the 2024-25 actual already in delivery
actual = B["lt120"]["Total net additional dwellings"].dropna()
actual.index = actual.index.map(fy_start)
if 2024 not in actual.index:
    actual.loc[2024] = rdl["2024-25"]
actual = actual.sort_index()

# Forecast path (2025-26 onward)
def fcast_series(d):
    s = pd.Series({fy_start(k): v for k, v in d.items() if fy_start(k) >= 2025}).sort_index()
    return pd.concat([actual.iloc[[-1]], s])   # prepend last actual to close the gap

# Prepend the last actual point so the forecast line connects without a gap
join_rdl  = fcast_series(rdl)
join_vecm = fcast_series(vecm)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(actual.index, actual.values, color="#1f4e79", lw=2, label="Actual")
ax.plot(join_rdl.index,  join_rdl.values,  color="#c0392b", lw=2, ls="--",
        marker="o", label="ARDL/NARDL ensemble (OBR-conditioned)")
ax.plot(join_vecm.index, join_vecm.values, color="#e08e0b", lw=2, ls=":",
        marker="s", label="VECM (unconditional system)")
ax.set_ylabel("Net additional dwellings")
ax.set_xlabel("Financial year (start)")
ax.legend()
plt.tight_layout()
plt.show()

# Chronos

In [ ]:
target = 1_500_000
B = build_bridge_inputs()

fc_chr = pd.read_csv("../../data/outputs/forecasts/chronos_forward.csv")
fc_chr["log_starts"] = np.log(fc_chr["starts"])
fc_chr = fc_chr.rename(columns={"Quarter": "period"})

chronos = run_bridge(fc_chr, "log_starts",
                      B["seed"], B["p"], B["seasonal"], B["fy_map"], B["net_add"], B["actual_back"])

print("--- Chronos (EXCLUDED from forward projection, shown only to illustrate why) ---")
for fy, v in chronos.items():
    print(f"{fy}: {v:,.0f}")

# Demonstration of staircase
print()
print(fc_chr[["period", "starts"]].to_string(index=False))